In [1]:
%load_ext autoreload
%autoreload 2
from src.data_utils import DataUtils
import yaml
import pandas as pd


In [2]:
# Загрузка конфига
with open("configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [22]:
from src.next_token_dataset import NextTokenDataset
from transformers import BertTokenizerFast
from torch.utils.data import DataLoader
import torch
from src.lstm_model import LstmModel
from transformers import pipeline

# Создание csv файлов
# DataUtils.samples_create(config['dataset'])

In [15]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

df_train = pd.read_csv(config['dataset']['path'] + '/train.csv')
df_val = pd.read_csv(config['dataset']['path'] + '/val.csv')

# train_dataset = NextTokenDataset(df_train, tokenizer, max_len=16)
# val_dataset = NextTokenDataset(df_val, tokenizer, max_len=16)

train_dataset = NextTokenDataset(df_train["text"].tolist(), tokenizer, max_len=16)
val_dataset = NextTokenDataset(df_val["text"].tolist(), tokenizer, max_len=16)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)


In [16]:
for batch in train_loader:
    print(batch["input_ids"].shape)
    break

torch.Size([256, 15])


In [17]:
from src.eval_transformer_pipeline import eval_transformer_pipeline
from src.utils import my_device

model = LstmModel(vocab_size=tokenizer.vocab_size, hidden_dim=128).to(my_device())

eval_transformer_pipeline(
    config=config,
    model=model,
    tokenizer=tokenizer,
    train_loader=train_loader,
    val_loader=val_loader
)

Epoch 1/10 | Train Loss: 4.8766 | Val Loss: 4.5478 | Val Acc: 0.3235
Epoch 2/10 | Train Loss: 4.4416 | Val Loss: 4.4047 | Val Acc: 0.3348
Epoch 3/10 | Train Loss: 4.3367 | Val Loss: 4.3678 | Val Acc: 0.3367
Epoch 4/10 | Train Loss: 4.2761 | Val Loss: 4.3127 | Val Acc: 0.3428
Epoch 5/10 | Train Loss: 4.2250 | Val Loss: 4.2865 | Val Acc: 0.3458
Epoch 6/10 | Train Loss: 4.1971 | Val Loss: 4.3203 | Val Acc: 0.3401
Epoch 7/10 | Train Loss: 4.1742 | Val Loss: 4.2624 | Val Acc: 0.3491
Epoch 8/10 | Train Loss: 4.1559 | Val Loss: 4.2846 | Val Acc: 0.3455
Epoch 9/10 | Train Loss: 4.1397 | Val Loss: 4.2450 | Val Acc: 0.3512
Epoch 10/10 | Train Loss: 4.1213 | Val Loss: 4.2405 | Val Acc: 0.3521


In [ ]:
model.eval()
for i in range(10):
    text = df_train.iloc[i]
    if isinstance(text, (pd.Series, dict)):
        text = text[0]  # если DataFrame с одной колонкой
    prompt = text.split()[: len(text.split()) * 3 // 4]
    prompt_str = " ".join(prompt)

    # Токенизируем "частичный" текст
    input_ids = tokenizer.encode(prompt_str, return_tensors="pt", truncation=True, max_length=32)

    # Генерируем дополнение
    generated_ids = model.generate(input_ids, max_new_tokens=10)
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print(f"🟢 Original: {text}")
    print(f"🔹 Prompt: {prompt_str}")
    print(f"🔸 Generated: {generated_text}\n{'-'*80}")


/var/folders/2p/1g0tbznd1zg9118y89qrclsc0000gn/T/ipykernel_10697/2919827760.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  text = text[0]  # если DataFrame с одной колонкой


🟢 Original: hanrudman aah lovely say hello to the ntw peeps swam in the pool sea paddling only but tonight ill swim in the sea for sure
🔹 Prompt: hanrudman aah lovely say hello to the ntw peeps swam in the
🔸 Generated: hanrudman aah lovely say hello to the ntw peeps swam in the
--------------------------------------------------------------------------------
🟢 Original: cjcroll why are you sad
🔹 Prompt: cjcroll why
🔸 Generated: cjcroll why
--------------------------------------------------------------------------------
🟢 Original: beyonces music is amazinggg lt3 love her ugh im so tired about to go to bed hope everyone has a good day tomorrow night sillys
🔹 Prompt: beyonces music is amazinggg lt3 love her ugh im so tired about
🔸 Generated: beyonces music is amazinggg lt3 love her ugh im so tired about
--------------------------------------------------------------------------------
🟢 Original: pbjcreations oh no what happened will sure pray for ya i know mamas hate when babies are hurtin

In [ ]:
# Тестирование моделей
from transformers import AutoTokenizer
from src.transformer import transformer_make_prediction


df_test = pd.read_csv(config['dataset']['path'] + '/test.csv')

generator = pipeline("text-generation", model="distilgpt2")
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
# test_dataset = NextTokenDataset(df_test["text"].tolist(), tokenizer, max_len=16)
# test_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

for i in range(10):
    text = df_test.iloc[i]["text"]
    prompt = text.split()[: len(text.split()) * 3 // 4]
    prompt_str = " ".join(prompt)

    # Модель distilgpt2
    print('PROMPT: ' + prompt_str)
    transformer_make_prediction(generator,tokenizer, text)
    print('-'*80)



Device set to use mps:0


trns:jonnyboyslim hah that would be a funny sight sleep time mark is makin me go to town v early tomorrow for last minute shoppin night xo
--------------------------------------------------------------------------------
trns:christinemc0828 yes i will admit i was pleasantly surprised with the quality of everything else in the trailer.    I think it is a bit of a surprise that the trailer is going to
--------------------------------------------------------------------------------
trns:cubs game today box seats cant wait but really wish zach was here.  The next big thing is the size of the table, the size of the table,
--------------------------------------------------------------------------------
trns:wikisignpost ive never seen wikipedia in g news listingssomething brand new not surprised theyve begun listing blogs in news listings   The current listing of the new Wikipedia site is in the "Wikipedia Wik
--------------------------------------------------------------------------------
t